# Простейшие нейронные сети на PyTorch

Цель семинара — обучить первую MLP и **корректно сравнить её с бейзлайнами по единой метрике**.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use('ggplot')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data_utils

print("PyTorch version:", torch.__version__)

## 1. О данных и бэйзлайнах

Наша главная цель - познакомиться с процессом создания простой нейронной сети и ее обучения.

__Делать всё это мы будем на животных.__ Ежегодно около 7.6 миллионов бедных животных в США оказываются в приютах. Часть из них находит себе новую семью, часть возвращается к старому (бывает, что питомец потерялся и его нашли на улице), а часть погибает. Ужегодно усыпляется около 2.7 млн. собак и кошек.  

Используя датасет с входной информацией (цвет, пол, возраст и т.п.) из одного из приютов, мы попытаемся спрогнозировать что произойдёт с новыми животными, которые попадут в этот приют. Данные, используемые в тетрадке уже были предварительно обработаны и приведены в удобную для построения моделей форму. 

https://www.kaggle.com/c/shelter-animal-outcomes

https://github.com/jakesherman/shelter-animals-kaggle/blob/master/4-modeling.ipynb

## 1. Загрузка и подготовка данных

In [ ]:
X = pd.read_csv('./data/X_cat.csv', sep='\t', index_col=0)
target = pd.read_csv('./data/y_cat.csv', sep='\t', index_col=0, names=['status'])

In [ ]:
print(X.shape)
X.head()

In [ ]:
X.head().T

In [ ]:
target.iloc[:, :]

In [ ]:
target['status'].unique()

В датасете находится около 27 тысяч наблюдений и 37 регрессоров. Посмотрим на то как выглядит распределение того, что произошло со зверятами по особям.

In [ ]:
target.status.value_counts()

Видим, что классы несбалансированы. Попробуем оставить четыре класса и объединить класс умерших животных с классом животных, которых усыпили. 

In [ ]:
# Объединяем 'Died' и 'Euthanasia'
target = target.replace('Died', 'Euthanasia')

In [ ]:
# Кодируем метки
le = LabelEncoder()
y = le.fit_transform(target['status'])
y


In [ ]:
# Разбиение
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [ ]:
help(train_test_split)

## 2. Бейзлайны (все оцениваются по log loss)

In [ ]:
# Константный прогноз
probas_const = pd.Series(y_train).value_counts(normalize=True).sort_index().values
constant_forecast = np.tile(probas_const, len(y_test)).reshape(len(y_test), -1)
logloss_const = log_loss(y_test, constant_forecast)


In [ ]:
logloss_const

In [ ]:

# Логистическая регрессия
logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train, y_train)
logloss_logreg = log_loss(y_test, logreg.predict_proba(X_test))


In [ ]:
logloss_logreg 

In [ ]:

# Случайный лес
rf = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
logloss_rf = log_loss(y_test, rf.predict_proba(X_test))
logloss_rf

In [ ]:

print(f"Constant forecast log loss: {logloss_const:.4f}")
print(f"Logistic Regression log loss: {logloss_logreg:.4f}")
print(f"Random Forest log loss: {logloss_rf:.4f}")

## 3. Нейронная сеть (оцениваем  по log loss)

In [ ]:
# Подготовка DataLoader
BATCH_SIZE = 128

def create_loader(X, y, batch_size, shuffle):
    dataset = data_utils.TensorDataset(
        torch.tensor(X.values.astype(np.float32)),
        torch.tensor(y.astype(np.int64))
    )
    return data_utils.DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

train_loader = create_loader(X_train, y_train, BATCH_SIZE, True)
test_loader = create_loader(X_test, y_test, BATCH_SIZE, False)

In [ ]:
# Архитектура БЕЗ Softmax!
INPUT_SIZE = X.shape[1]
HIDDEN_SIZE = 25
OUTPUT_SIZE = len(np.unique(y))

model = nn.Sequential(
    nn.Linear(INPUT_SIZE, HIDDEN_SIZE),
    nn.ReLU(),
    nn.Linear(HIDDEN_SIZE, HIDDEN_SIZE),
    nn.ReLU(),
    nn.Linear(HIDDEN_SIZE, OUTPUT_SIZE)
    # ← НЕТ Softmax! CrossEntropyLoss сам применяет log-softmax
)

In [ ]:
# Обучение
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 100
train_losses, val_losses = [], []

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    train_losses.append(epoch_loss / len(train_loader))

    # Валидация
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            val_loss += loss.item()
    val_losses.append(val_loss / len(test_loader))

    if epoch % 20 == 0:
        print(f"Epoch {epoch}: train loss = {train_losses[-1]:.4f}, val loss = {val_losses[-1]:.4f}")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label='Train Loss', color='tab:blue')
plt.plot(val_losses, label='Validation Loss', color='tab:orange')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Оценка по log loss
model.eval()
all_logits = []
with torch.no_grad():
    for X_batch, _ in test_loa der:
        logits = model(X_batch)
        all_logits.append(logits)

logits_test = torch.cat(all_logits, dim=0)
probas_nn = torch.softmax(logits_test, dim=1).numpy()
logloss_nn = log_loss(y_test, probas_nn)

print(f"Neural Network log loss: {logloss_nn:.4f}")

## 4. Сравнение всех моделей

In [ ]:

results = pd.DataFrame({
    "Model": ["Constant", "Logistic Regression", "Random Forest", "Neural Network"],
    "Log Loss (test)": [logloss_const, logloss_logreg, logloss_rf, logloss_nn]
}).sort_values("Log Loss (test)")

results

1. Начальный скачок loss
   
В нулевой эпохе train_loss ≈ 3.5 — это значение initial cross-entropy при случайных весах (для 4 классов: -log(1/4) ≈ 1.386, но у вас выше — возможно, из-за несбалансированности или плохой инициализации).
Резкое падение на первых эпохах — нормально: модель быстро учится «грубым» шаблонам.
   
2. Разрыв между train и val loss
После ~20 эпох:
train_loss ≈ 0.85,
val_loss ≈ 0.95–1.0
→ разница ~0.1–0.15 — это небольшое переобучение, но не критичное.

3. Стагнация после 40 эпох
Оба loss’а почти перестают снижаться после 40–50 эпох — обучение зашло в плато.
Это означает, что:
либо модель достигла предела своей выразительной способности при текущих гиперпараметрах,
либо learning rate слишком мал для дальнейшего прогресса.


# Что дальше?

1. nn.ReLU() → nn.LeakyReLU(0.1)
2. HIDDEN_SIZE = 25 → 64 или 128
3. Добавить drop out
   model = nn.Sequential(
    nn.Linear(INPUT_SIZE, 128),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(64, OUTPUT_SIZE)
)
4. optimizer = optim.Adam(model.parameters(), lr=5e-4)  # вместо 1e-3
5.  Увеличьте число эпох до 200–300, но добавьте раннюю остановку

Увеличьте HIDDEN_SIZE = 64 или добавьте третий слой.
Попробуйте lr = 5e-4 вместо 1e-3.
Добавьте Dropout(0.3) после ReLU — если val loss начнёт расти, уберите.

In [ ]:
# model = nn.Sequential(
#     nn.Linear(INPUT_SIZE, 128),
#     nn.ReLU(),
#     nn.Dropout(0.3),
#     nn.Linear(128, 64),
#     nn.ReLU(),
#     nn.Dropout(0.3),
#     nn.Linear(64, OUTPUT_SIZE)
# )
# optimizer = optim.Adam(model.parameters(), lr=5e-4)
# EPOCHS = 150